In [2]:
# Install ───────────────────────────────────────────────────────────
!pip install transformers datasets scikit-learn -q

In [3]:
#  Upload labels.csv ─────────────────────────────────────────────────
from google.colab import files
uploaded = files.upload()   # загрузи labels.csv

Saving labels.csv to labels.csv


In [4]:
#  Prepare dataset ───────────────────────────────────────────────────
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split

LABELS = ["noise", "important", "error", "security_noise"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

df = pd.read_csv("labels.csv")
df = df[df["label"].isin(LABELS)].reset_index(drop=True)
df["label_id"] = df["label"].map(LABEL2ID)

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["label"]
)

train_ds = Dataset.from_pandas(train_df[["template", "label_id"]])
test_ds  = Dataset.from_pandas(test_df[["template",  "label_id"]])

print(f"Train: {len(train_ds)}, Test: {len(test_ds)}")
print(dict(df["label"].value_counts()))

# Tokenize ──────────────────────────────────────────────────────────
from transformers import AutoTokenizer

# LogBERT uses bert-base-uncased as backbone
MODEL_NAME = "bert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["template"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

train_ds = train_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize,  batched=True)

train_ds = train_ds.rename_column("label_id", "labels")
test_ds  = test_ds.rename_column("label_id",  "labels")
train_ds.set_format("torch", columns=["input_ids","attention_mask","labels"])
test_ds.set_format("torch",  columns=["input_ids","attention_mask","labels"])



Train: 51795, Test: 12949
{'noise': np.int64(45533), 'important': np.int64(11002), 'error': np.int64(6937), 'security_noise': np.int64(1272)}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/51795 [00:00<?, ? examples/s]

Map:   0%|          | 0/12949 [00:00<?, ? examples/s]

In [5]:
#  Train ─────────────────────────────────────────────────────────────
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
import numpy as np
from sklearn.metrics import f1_score

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "accuracy": (preds == labels).mean(),
    }

training_args = TrainingArguments(
    output_dir          = "logbert_anomalens",
    num_train_epochs    = 5,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size  = 64,
    warmup_steps        = 100,
    weight_decay        = 0.01,
    eval_strategy       = "epoch",
    save_strategy       = "best",
    load_best_model_at_end = True,
    metric_for_best_model  = "macro_f1",
    fp16                = True,   # faster on T4
    report_to           = "none",
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = test_ds,
    compute_metrics = compute_metrics,
)

trainer.train()



model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.107094,0.106572,0.942659,0.949880
2,0.096998,0.093961,0.956155,0.956445
3,0.092788,0.091294,0.945456,0.951270
4,0.087911,0.091608,0.956652,0.956367
5,0.084248,0.091511,0.956802,0.956522


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=8095, training_loss=0.10433627582759751, metrics={'train_runtime': 1774.7551, 'train_samples_per_second': 145.922, 'train_steps_per_second': 4.561, 'total_flos': 1.70351022875904e+16, 'train_loss': 0.10433627582759751, 'epoch': 5.0})

In [6]:
# Evaluate and save ─────────────────────────────────────────────────
results = trainer.evaluate()
print(f"\nFinal macro F1: {results['eval_macro_f1']:.4f}")
print(f"Final accuracy: {results['eval_accuracy']:.4f}")

# Full classification report
from sklearn.metrics import classification_report
preds_output = trainer.predict(test_ds)
preds = np.argmax(preds_output.predictions, axis=-1)
print(classification_report(
    test_ds["labels"], preds,
    target_names=LABELS, zero_division=0
))




Final macro F1: 0.9568
Final accuracy: 0.9565
                precision    recall  f1-score   support

         noise       0.99      0.95      0.97      9107
     important       0.82      1.00      0.90      2201
         error       1.00      0.95      0.97      1387
security_noise       0.98      1.00      0.99       254

      accuracy                           0.96     12949
     macro avg       0.95      0.97      0.96     12949
  weighted avg       0.96      0.96      0.96     12949



In [7]:
# Save model
model.save_pretrained("logbert_anomalens_final")
tokenizer.save_pretrained("logbert_anomalens_final")

# Download
import shutil
shutil.make_archive("logbert_anomalens_final", "zip", "logbert_anomalens_final")
files.download("logbert_anomalens_final.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

To speedup LogBERT on CPU - convertation to ONNX

In [8]:
'''
!pip install optimum onnxruntime -q

from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

# Convert to ONNX
ort_model = ORTModelForSequenceClassification.from_pretrained(
    "logbert_anomalens_final",
    export=True,
)
ort_model.save_pretrained("logbert_anomalens_onnx")
tokenizer.save_pretrained("logbert_anomalens_onnx")
'''

'\n!pip install optimum onnxruntime -q\n\nfrom optimum.onnxruntime import ORTModelForSequenceClassification\nfrom transformers import AutoTokenizer\n\n# Convert to ONNX\nort_model = ORTModelForSequenceClassification.from_pretrained(\n    "logbert_anomalens_final",\n    export=True,\n)\nort_model.save_pretrained("logbert_anomalens_onnx")\ntokenizer.save_pretrained("logbert_anomalens_onnx")\n'

In [9]:

# download
'''
import shutil
shutil.make_archive("logbert_anomalens_onnx", "zip", "logbert_anomalens_onnx")
files.download("logbert_anomalens_onnx.zip")
'''

'\nimport shutil\nshutil.make_archive("logbert_anomalens_onnx", "zip", "logbert_anomalens_onnx")\nfiles.download("logbert_anomalens_onnx.zip")\n'

Use for evaluation:

In [10]:
'''
from optimum.onnxruntime import ORTModelForSequenceClassification

self.classifier = hf_pipeline(
    "text-classification",
    model=ORTModelForSequenceClassification.from_pretrained(model_path),
    tokenizer=AutoTokenizer.from_pretrained(model_path),
    device=-1,
    truncation=True,
    max_length=128,
)
'''

'\nfrom optimum.onnxruntime import ORTModelForSequenceClassification\n\nself.classifier = hf_pipeline(\n    "text-classification",\n    model=ORTModelForSequenceClassification.from_pretrained(model_path),\n    tokenizer=AutoTokenizer.from_pretrained(model_path),\n    device=-1,\n    truncation=True,\n    max_length=128,\n)\n'